# Pseudobulk of neural crest cells in zebrafish

In [1]:
import scanpy as sc
import pandas as pd

pd.options.display.max_columns = None

## Load the raw data

In [2]:
# downloaded from https://www.heartcellatlas.org/ # Heart Global, raw
ad = sc.read('/hpc/mydata/mathias.voges/Projects/research/zf-decima/data/zf_atlas_neuralcrest_v4_release.h5ad')

In [5]:
#ad.obs.region_finest = ad.obs.region_finest.astype(str)

In [6]:
# ad.obs.loc[ad.obs.region_finest == 'na', 'region_finest'] = 'SAN_unknown'
# ad.obs.loc[ad.obs.region_finest == 'IVS MID LV', 'region_finest'] = 'SP IVS MID LV'
# ad.obs.loc[ad.obs.region_finest == 'IVS MID RV', 'region_finest'] = 'SP IVS MID RV'

In [7]:
ad._sanitize()

## Filtering 

In [22]:
ad.obs

,annotation,timepoint,fish,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,total_counts_nc,pct_counts_nc
TDR70_ACGTAACCAACTGATC-1,oligodendrocyte,3dpf,TDR70,1466,5203.0,406.0,7.803191,411.0,7.899289
TDR46_GTTCATTGTTTGAAAG-1,retinal_pigmented_epithelium,24hpf,TDR46,3820,20816.0,481.0,2.310723,261.0,1.253843
TDR48_GTCTACCTCCTCTCGA-1,xanthoblast,2dpf,TDR48,811,2157.0,133.0,6.165971,191.0,8.854892
TDR52_CACTGGGAGCGTTAGG-1,oligodendrocyte,5dpf,TDR52,1282,3647.0,238.0,6.525911,175.0,4.798464
TDR67_TAGCACAAGGTACATA-1,oligodendrocyte,3dpf,TDR67,1689,5700.0,246.0,4.315789,408.0,7.157895
...,...,...,...,...,...,...,...,...,...
TDR54_TTAGTCTGTGACTGAG-1,oligodendrocyte,5dpf,TDR54,2330,9905.0,295.0,2.978294,318.0,3.210500
TDR51_CATGGTAGTATCGCTA-1,oligodendrocyte,5dpf,TDR51,1985,4478.0,272.0,6.074140,232.0,5.180884
TDR47_TCATATCGTCTTCGAA-1,oligodendrocyte,2dpf,TDR47,1846,6205.0,212.0,3.416599,177.0,2.852538
TDR44_CATGCAATCCACGGAC-1,unassigned,24hpf,TDR44,4045,22950.0,407.0,1.773421,370.0,1.612200


In [3]:
ident_cols = ['annotation']  # adjust these columns as needed
counts = ad.obs[ident_cols].value_counts().reset_index()
counts = counts.rename(columns={'count': 'n_cells'})


In [4]:
adp = sc.get.aggregate(ad, ident_cols, func='sum')


In [5]:
adp.X = adp.layers['sum'].astype(int)


In [6]:
del adp.layers['sum']  # cleanup
adp.obs = adp.obs.merge(counts, how='left')

In [14]:
adp.obs

,annotation,n_cells
0,brain_oligodendrocyte,164
1,iridophore,149
2,oligodendrocyte,820
3,retinal_pigmented_epithelium,170
4,unassigned,315
5,xanthoblast,346
6,xanthophore,213


In [16]:
adp.X

array([[   0,    0,    0, ...,  693,    1,    0],
       [   0,    0,    0, ...,  552,    0,    0],
       [   6,    0,    6, ..., 3020,    2,    0],
       ...,
       [   0,    0,    0, ..., 1138,    1,    0],
       [   0,    0,    1, ..., 1295,    0,    1],
       [   0,    0,    1, ...,  813,    0,    0]])

In [19]:
#ad = ad[(ad.obs.cell_or_nuclei == 'Nuclei') &  (ad.obs.cell_state!='unclassified')].copy()

In [ ]:
#ad.obs.cell_state = ad.obs.cell_type.astype(str) + '-' + ad.obs.cell_state.astype(str)

In [ ]:
#ident_cols = ['sample_ID', 'region_finest', 'cell_state', 'cell_type']
#ad.obs = ad.obs[ident_cols].copy()

In [ ]:
#for c in ident_cols:
#    ad.obs[c] = ad.obs[c].astype(str)

In [9]:
ad._sanitize()

## Now the pseudobulking

In [12]:
ad.var

,mean,n_cells,nc,n_cells_by_counts,mt,genome,mean_counts,variance,feature_types,gene_ids,total_counts,pct_dropout_by_counts,log1p_mean_counts,log1p_total_counts
ptpn12,0.004849,1552,False,22,False,Danio_rerio_genome_Zebrabow_6,0.010565,0.003064,Gene Expression,ENSDARG00000102141,23.0,98.989435,0.010510,3.178054
phtf2,0.000643,195,False,2,False,Danio_rerio_genome_Zebrabow_6,0.000919,0.000422,Gene Expression,ENSDARG00000102123,2.0,99.908130,0.000918,1.098612
phtf2-1,0.009970,2821,False,25,False,Danio_rerio_genome_Zebrabow_6,0.011484,0.007035,Gene Expression,ENSDARG00000114503,25.0,98.851631,0.011418,3.258096
CU856344.1,0.000222,44,False,1,False,Danio_rerio_genome_Zebrabow_6,0.000459,0.000203,Gene Expression,ENSDARG00000115971,1.0,99.954065,0.000459,0.693147
si:zfos-932h1.3,0.007119,2299,False,32,False,Danio_rerio_genome_Zebrabow_6,0.015618,0.004368,Gene Expression,ENSDARG00000098311,34.0,98.530087,0.015497,3.555348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mt-nd6,0.280758,48576,False,765,True,Danio_rerio_genome_Zebrabow_6,0.605880,0.182445,Gene Expression,ENSDARG00000063922,1319.0,64.859899,0.473672,7.185387
NC_002333.21,0.000143,49,True,0,False,Danio_rerio_genome_Zebrabow_6,0.000000,0.000080,Gene Expression,ENSDARG00000083312,0.0,100.000000,0.000000,0.000000
mt-cyb,3.609797,120579,False,2177,True,Danio_rerio_genome_Zebrabow_6,44.339916,0.475931,Gene Expression,ENSDARG00000063924,96528.0,0.000000,3.814188,11.477599
NC_002333.22,0.003168,943,True,15,False,Danio_rerio_genome_Zebrabow_6,0.006890,0.002108,Gene Expression,ENSDARG00000083462,15.0,99.310978,0.006867,2.772589


In [ ]:
counts = ad.obs[ident_cols].value_counts().reset_index()
counts = counts.rename(columns={'count': 'n_cells'})

In [10]:
ad = ad[ad.obs.merge(counts, how='left').n_cells>=10].copy()

NameError: name 'counts' is not defined

In [ ]:
sc.pp.filter_genes(ad, min_cells=50)

In [ ]:
ad.var['gene_id'] = ad.var.index
ad.var.index = ad.var['gene_name-new']
ad.var.index.name = None

In [ ]:
counts = ad.obs[ident_cols].value_counts().reset_index()
counts = counts.rename(columns={'count': 'n_cells'})

In [ ]:
adp = sc.get.aggregate(ad, ident_cols, func='sum')

In [ ]:
adp.X = adp.layers['sum'].astype(int)

In [ ]:
del adp.layers['sum']

### Add cell counts

In [ ]:
adp.obs = adp.obs.merge(counts, how='left')

In [ ]:
adp.obs.rename(columns={'region_finest': 'region'}, inplace=True)

In [ ]:
adp = adp.copy()

## Save

In [ ]:
adp.write('heart-pseudobulk.h5ad')